In [20]:
!pip install transformers --upgrade

In [21]:
import pandas as pd

# 파일을 읽어와서 데이터의 처음 5행을 출력한다.
reviews = pd.read_csv('./data/review.csv')
reviews.head()


,name,star,date,content,utility,g_no
0,지서원,4,2023년 6월 26일,게임도 매우 재밌고 창의적입니다 근데 스테이지를 올라가면 갈수록 난이도는 점점 어려...,213,1
1,pori,3,2023년 7월 2일,300스테이지 이상 하고 있는 유저인데 현질 유도가 엄청 심하네요.. 그렇다고 저렴...,90,1
2,Google 사용자,1,2023년 6월 29일,분명히 게임 잘 하고있었습니다. 캔디러시?러쉬? 2판 남겨두고 오랜만에 현질하려니 ...,12,1
3,강희야,4,2023년 7월 19일,재미잇어서 오랫동안 하고있는 유일한 게임입니다. 요즘 캔디로얄 시작버튼만있고 x버튼...,0,1
4,조정은,4,2023년 6월 25일,유일하게하는게임입니다 아이템없이는 레벨업이 힘들어요 유료템을 구입하는것도 한계가있는...,173,1


In [22]:
from transformers import BertTokenizer,BertForSequenceClassification
from torch.nn.functional import softmax
import torch

# Pre-trained BERT model for sentiment analysis
MODEL_NAME = "nlptown/bert-base-multilingual-uncased-sentiment"
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model = BertForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()

def predict_sentiment(text):
    """
    Predicts the sentiment of a given text using the BERT model.
    Returns 'positive' if the sentiment is positive, and 'negative' otherwise.
    """
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = softmax(outputs.logits, dim=1)
    score = torch.argmax(probs, dim=1).item()
    
    # The model returns a score between 1 to 5 (it was originally trained for star rating prediction).
    # We'll consider scores 1-2 as negative and 4-5 as positive. 3 will be considered neutral.
    if score in [1, 2]:
        return "negative"
    elif score in [4, 5]:
        return "positive"
    else:
        return "neutral"

# Apply sentiment prediction on the review contents
reviews["sentiment"] = reviews["content"].apply(predict_sentiment)
reviews[["content", "sentiment"]].head()


ImportError: cannot import name 'BertForSequenceClassification' from 'transformers' (C:\Users\user07\anaconda3\Lib\site-packages\transformers\__init__.py)